### Step 1 — Install packages

In [1]:
# Step 1: Install dependencies

!pip install -q wandb optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 9.5 MB/s eta 0:00:00


### Step 2 — Mount Drive and imports

In [19]:
# Step 2: Mount Google Drive and import packages

from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

import os
import json
import time
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

import wandb
import optuna

torch.backends.cudnn.benchmark = True

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Step 3 — Define paths

In [20]:
# Step 3: Define paths

BASE_DIR = Path("/content/drive/MyDrive/rectified_flow_ct2dose")
SPLIT_DIR = BASE_DIR / "data" / "splits"
CKPT_DIR = BASE_DIR / "outputs" / "checkpoints"
OUT_DIR = BASE_DIR / "analysis" / "phase3_wandb_optuna"

OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_MANIFEST = SPLIT_DIR / "train_pairs_3d_train_phase3_2000.json"
VAL_MANIFEST = SPLIT_DIR / "train_pairs_3d_val_phase3_500.json"

print("BASE_DIR exists:", BASE_DIR.exists())
print("TRAIN_MANIFEST exists:", TRAIN_MANIFEST.exists())
print("VAL_MANIFEST exists:", VAL_MANIFEST.exists())
print("OUT_DIR exists:", OUT_DIR.exists())

BASE_DIR exists: True
TRAIN_MANIFEST exists: True
VAL_MANIFEST exists: True
OUT_DIR exists: True


### Step 4 — Reuse verified definitions

In [21]:
# Step 4: Paste the following verified definitions here
# - CubePair3DDataset
# - ConditionalUNetFlow3D
# - euler_sample_flow_3d
# - compute_flow_matching_loss
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

def normalize_ct(x):
    x = np.clip(x, -1024.0, 1500.0)
    x = (x + 1024.0) / (1500.0 + 1024.0)
    return x.astype(np.float32)

class CubePair3DDataset(Dataset):
    def __init__(self, manifest_path, dose_scale=1000.0):
        with open(manifest_path, "r", encoding="utf-8") as f:
            self.records = json.load(f)
        self.dose_scale = dose_scale

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]

        x0 = np.load(rec["input_path"]).astype(np.float32)
        x1 = np.load(rec["output_path"]).astype(np.float32)

        x0 = normalize_ct(x0)
        x1 = (x1 * self.dose_scale).astype(np.float32)

        x0 = torch.from_numpy(x0).unsqueeze(0)   # (1,D,H,W)
        x1 = torch.from_numpy(x1).unsqueeze(0)   # (1,D,H,W)

        return x0, x1


class DoubleConv3D(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)



class ConditionalUNetFlow3D(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, base_ch=24):
        super().__init__()

        self.enc1 = DoubleConv3D(in_ch, base_ch)
        self.pool1 = nn.MaxPool3d(2)

        self.enc2 = DoubleConv3D(base_ch, base_ch * 2)
        self.pool2 = nn.MaxPool3d(2)

        self.bottleneck = DoubleConv3D(base_ch * 2, base_ch * 4)

        self.up2 = nn.ConvTranspose3d(base_ch * 4, base_ch * 2, kernel_size=2, stride=2)
        self.dec2 = DoubleConv3D(base_ch * 4, base_ch * 2)

        self.up1 = nn.ConvTranspose3d(base_ch * 2, base_ch, kernel_size=2, stride=2)
        self.dec1 = DoubleConv3D(base_ch * 2, base_ch)

        self.out_conv = nn.Conv3d(base_ch, out_ch, kernel_size=1)

    def forward(self, xt, x0, t):
        # xt, x0: (B,1,D,H,W)
        # t:      (B,1,1,1,1)
        t_map = t.expand(-1, 1, xt.shape[2], xt.shape[3], xt.shape[4])  # (B,1,D,H,W)
        inp = torch.cat([xt, x0, t_map], dim=1)  # (B,3,D,H,W)

        e1 = self.enc1(inp)
        e2 = self.enc2(self.pool1(e1))
        b  = self.bottleneck(self.pool2(e2))

        d2 = self.up2(b)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.out_conv(d1)  # predicted velocity


@torch.no_grad()
def euler_sample_flow_3d(model, x0, n_steps=30):
    """
    x0: (B,1,D,H,W)
    """
    model.eval()
    z = x0.clone()
    dt = 1.0 / n_steps

    for k in range(n_steps):
        t = torch.full((z.shape[0], 1, 1, 1, 1), k / n_steps, device=z.device)
        v = model(z, x0, t)
        z = z + dt * v

    return z


def sample_t_like(x):
    return torch.rand((x.shape[0], 1, 1, 1, 1), device=x.device)


def compute_flow_matching_loss(model, x0, x1):
    t = sample_t_like(x0)
    z_t = (1.0 - t) * x0 + t * x1
    v_target = x1 - x0
    v_pred = model(z_t, x0, t)
    return F.mse_loss(v_pred, v_target)

### Step 5 — Set seeds and device

In [22]:
# Step 5: Set seeds and device

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: NVIDIA A100-SXM4-40GB


### Step 6 — Build datasets

In [23]:
# Step 6: Build datasets

DOSE_SCALE = 1000.0
NUM_WORKERS = 2

train_dataset_3d = CubePair3DDataset(TRAIN_MANIFEST, dose_scale=DOSE_SCALE)
val_dataset_3d = CubePair3DDataset(VAL_MANIFEST, dose_scale=DOSE_SCALE)

print("Train dataset length:", len(train_dataset_3d))
print("Val dataset length:", len(val_dataset_3d))

Train dataset length: 2000
Val dataset length: 500


### Step 7 — Helper functions

In [24]:
# Step 7: Helper functions

def make_loaders(batch_size):
    train_loader = DataLoader(
        train_dataset_3d,
        batch_size=batch_size,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0)
    )

    val_loader = DataLoader(
        val_dataset_3d,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0)
    )

    return train_loader, val_loader


def train_one_epoch_flow(model, loader, optimizer, device, scaler):
    model.train()

    total_loss = 0.0
    n_batches = 0

    for x0, x1 in loader:
        x0 = x0.to(device, non_blocking=True)
        x1 = x1.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)


        amp_device = "cuda" if torch.cuda.is_available() else "cpu"
        with torch.amp.autocast(amp_device, enabled=torch.cuda.is_available()):
            loss = compute_flow_matching_loss(model, x0, x1)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        n_batches += 1

    return total_loss / max(n_batches, 1)


@torch.no_grad()
def evaluate_flow_matching_loss(model, loader, device):
    model.eval()

    total_loss = 0.0
    n_batches = 0

    for x0, x1 in loader:
        x0 = x0.to(device, non_blocking=True)
        x1 = x1.to(device, non_blocking=True)

        amp_device = "cuda" if torch.cuda.is_available() else "cpu"
        with torch.amp.autocast(amp_device, enabled=torch.cuda.is_available()):
            loss = compute_flow_matching_loss(model, x0, x1)

        total_loss += loss.item()
        n_batches += 1

    return total_loss / max(n_batches, 1)


@torch.no_grad()
def evaluate_flow_prediction_metrics(model, loader, device, n_steps=30):
    model.eval()

    total_mse = 0.0
    total_mae = 0.0
    n_batches = 0

    for x0, x1 in loader:
        x0 = x0.to(device, non_blocking=True)
        x1 = x1.to(device, non_blocking=True)

        pred = euler_sample_flow_3d(model, x0, n_steps=n_steps)

        total_mse += F.mse_loss(pred, x1).item()
        total_mae += F.l1_loss(pred, x1).item()
        n_batches += 1

    return {
        "mse": total_mse / max(n_batches, 1),
        "mae": total_mae / max(n_batches, 1),
    }


def save_checkpoint(path, model, optimizer, epoch, train_losses, val_losses, config):
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_losses": train_losses,
        "val_losses": val_losses,
        "config": config,
    }, path)

### Step 8 — Login to wandb

In [25]:
# Step 8: Login to wandb

wandb.login()

True

### Step 9 — Define one wandb run function

In [26]:
# Step 9: Define one wandb-enabled training function

def run_one_wandb_experiment(config, project_name="ct2dose-phase3-tuning"):
    run = wandb.init(
        project=project_name,
        config=config,
        reinit=True
    )

    cfg = wandb.config

    train_loader, val_loader = make_loaders(batch_size=cfg.batch_size)

    model = ConditionalUNetFlow3D(
        in_ch=3,
        out_ch=1,
        base_ch=cfg.base_ch
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)
    amp_device = "cuda" if torch.cuda.is_available() else "cpu"
    scaler = torch.amp.GradScaler(amp_device, enabled=torch.cuda.is_available())

    train_losses = []
    val_losses = []

    best_val_loss = float("inf")
    best_epoch = -1

    ckpt_latest = CKPT_DIR / f"{cfg.run_name}_latest.pt"
    ckpt_best = CKPT_DIR / f"{cfg.run_name}_best.pt"

    start_time = time.time()

    for epoch in range(1, cfg.epochs + 1):
        train_loss = train_one_epoch_flow(model, train_loader, optimizer, device, scaler)
        val_loss = evaluate_flow_matching_loss(model, val_loader, device)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        wandb.log({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
        })

        save_checkpoint(
            ckpt_latest,
            model,
            optimizer,
            epoch,
            train_losses,
            val_losses,
            dict(cfg)
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch

            save_checkpoint(
                ckpt_best,
                model,
                optimizer,
                epoch,
                train_losses,
                val_losses,
                dict(cfg)
            )

    best_ckpt = torch.load(ckpt_best, map_location="cpu")
    model.load_state_dict(best_ckpt["model_state_dict"])
    model.eval()

    final_metrics = evaluate_flow_prediction_metrics(model, val_loader, device, n_steps=30)

    elapsed_min = (time.time() - start_time) / 60.0

    summary = {
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "final_val_mse": final_metrics["mse"],
        "final_val_mae": final_metrics["mae"],
        "elapsed_min": elapsed_min,
        "best_ckpt_path": str(ckpt_best),
        "latest_ckpt_path": str(ckpt_latest),
    }

    wandb.log(summary)
    wandb.finish()

    return summary

### Step 10 — Smoke test

In [27]:
# Step 10: Smoke test

smoke_config = {
    "run_name": "wandb_smoke_lr3e4_base24_bs2_ep5",
    "lr": 3e-4,
    "base_ch": 24,
    "batch_size": 2,
    "epochs": 5,
}

smoke_summary = run_one_wandb_experiment(smoke_config)
smoke_summary

best_epoch,▁
best_val_loss,▁
elapsed_min,▁
epoch,▁▃▅▆█
final_val_mae,▁
final_val_mse,▁
train_loss,█▂▁▁▁
val_loss,█▃▁▂▁
best_ckpt_path,/content/drive/MyDri...
best_epoch,5
best_val_loss,0.00014


{'best_epoch': 5,
 'best_val_loss': 0.00013943385680613573,
 'final_val_mse': 0.00019689228030620143,
 'final_val_mae': 0.011017755195498467,
 'elapsed_min': 1.1747387290000915,
 'best_ckpt_path': '/content/drive/MyDrive/rectified_flow_ct2dose/outputs/checkpoints/wandb_smoke_lr3e4_base24_bs2_ep5_best.pt',
 'latest_ckpt_path': '/content/drive/MyDrive/rectified_flow_ct2dose/outputs/checkpoints/wandb_smoke_lr3e4_base24_bs2_ep5_latest.pt'}

### Step 11 — Run a small wandb tuning round

In [28]:
# Step 11: Small wandb tuning round

WANDB_CONFIGS = [
    {"run_name": "wandb_lr1e4_base24_bs2_ep30", "lr": 1e-4, "base_ch": 24, "batch_size": 2, "epochs": 30},
    {"run_name": "wandb_lr3e4_base24_bs2_ep30", "lr": 3e-4, "base_ch": 24, "batch_size": 2, "epochs": 30},
    {"run_name": "wandb_lr5e4_base24_bs2_ep30", "lr": 5e-4, "base_ch": 24, "batch_size": 2, "epochs": 30},
    {"run_name": "wandb_lr3e4_base32_bs2_ep30", "lr": 3e-4, "base_ch": 32, "batch_size": 2, "epochs": 30},
]

wandb_results = []

for cfg in WANDB_CONFIGS:
    result = run_one_wandb_experiment(cfg)
    row = dict(cfg)
    row.update(result)
    wandb_results.append(row)

wandb_results_df = pd.DataFrame(wandb_results).sort_values("best_val_loss", ascending=True).reset_index(drop=True)

wandb_csv_path = OUT_DIR / "wandb_round_results.csv"
wandb_results_df.to_csv(wandb_csv_path, index=False)

print("Saved:", wandb_csv_path)
wandb_results_df

best_epoch,▁
best_val_loss,▁
elapsed_min,▁
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
final_val_mae,▁
final_val_mse,▁
train_loss,█▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▄▃▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_ckpt_path,/content/drive/MyDri...
best_epoch,28
best_val_loss,3e-05


best_epoch,▁
best_val_loss,▁
elapsed_min,▁
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
final_val_mae,▁
final_val_mse,▁
train_loss,█▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▅▃▃▃▃▆▂▂▅▁▁▁▃▁▁▁▂▁▁▁▂▁▁▁▁▁▁▁▁
best_ckpt_path,/content/drive/MyDri...
best_epoch,29
best_val_loss,2e-05


best_epoch,▁
best_val_loss,▁
elapsed_min,▁
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
final_val_mae,▁
final_val_mse,▁
train_loss,█▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▃▃▂▂▁▂▂▂▁▁▂▂▁▁▁▁▁▁▂▁▁▁▁▁▁▁▂▁▁
best_ckpt_path,/content/drive/MyDri...
best_epoch,30
best_val_loss,2e-05


best_epoch,▁
best_val_loss,▁
elapsed_min,▁
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
final_val_mae,▁
final_val_mse,▁
train_loss,█▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▂▂▁▁▂▂▁▁▁▁▁▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_ckpt_path,/content/drive/MyDri...
best_epoch,28
best_val_loss,2e-05


Saved: /content/drive/MyDrive/rectified_flow_ct2dose/analysis/phase3_wandb_optuna/wandb_round_results.csv


,run_name,lr,base_ch,batch_size,epochs,best_epoch,best_val_loss,final_val_mse,final_val_mae,elapsed_min,best_ckpt_path,latest_ckpt_path
0,wandb_lr3e4_base24_bs2_ep30,0.0003,24,2,30,29,0.000019,0.000021,0.002897,5.966394,/content/drive/MyDrive/rectified_flow_ct2dose/...,/content/drive/MyDrive/rectified_flow_ct2dose/...
1,wandb_lr3e4_base32_bs2_ep30,0.0003,32,2,30,28,0.000020,0.000026,0.003401,6.224609,/content/drive/MyDrive/rectified_flow_ct2dose/...,/content/drive/MyDrive/rectified_flow_ct2dose/...
2,wandb_lr5e4_base24_bs2_ep30,0.0005,24,2,30,30,0.000021,0.000024,0.003295,5.978475,/content/drive/MyDrive/rectified_flow_ct2dose/...,/content/drive/MyDrive/rectified_flow_ct2dose/...
3,wandb_lr1e4_base24_bs2_ep30,0.0001,24,2,30,28,0.000029,0.000035,0.003695,5.969540,/content/drive/MyDrive/rectified_flow_ct2dose/...,/content/drive/MyDrive/rectified_flow_ct2dose/...


### Step 12 — Define Optuna objective

In [29]:
# Step 12: Define Optuna objective

def optuna_objective(trial):
    lr = trial.suggest_float("lr", 1e-4, 6e-4, log=True)
    base_ch = trial.suggest_categorical("base_ch", [24, 32])
    batch_size = trial.suggest_categorical("batch_size", [2])
    epochs = 20  # Keep the study smaller and faster

    train_loader, val_loader = make_loaders(batch_size=batch_size)

    model = ConditionalUNetFlow3D(
        in_ch=3,
        out_ch=1,
        base_ch=base_ch
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    amp_device = "cuda" if torch.cuda.is_available() else "cpu"
    scaler = torch.amp.GradScaler(amp_device, enabled=torch.cuda.is_available())

    best_val_loss = float("inf")

    for epoch in range(1, epochs + 1):
        _ = train_one_epoch_flow(model, train_loader, optimizer, device, scaler)
        val_loss = evaluate_flow_matching_loss(model, val_loader, device)

        if val_loss < best_val_loss:
            best_val_loss = val_loss

        trial.report(best_val_loss, step=epoch)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return best_val_loss

### Step 13 — Run Optuna study

In [30]:
# Step 13: Run Optuna study

study = optuna.create_study(direction="minimize")
study.optimize(optuna_objective, n_trials=8)

print("Best trial:")
print("Value:", study.best_trial.value)
print("Params:", study.best_trial.params)

[I 2026-04-12 10:01:40,466] A new study created in memory with name: no-name-664cebd1-aef7-4f0e-be35-b31edeb6a920
[I 2026-04-12 10:05:30,128] Trial 0 finished with value: 2.628667415046948e-05 and parameters: {'lr': 0.00015917718036895313, 'base_ch': 32, 'batch_size': 2}. Best is trial 0 with value: 2.628667415046948e-05.
[I 2026-04-12 10:09:18,528] Trial 1 finished with value: 3.714273853984196e-05 and parameters: {'lr': 0.00028806439560959346, 'base_ch': 32, 'batch_size': 2}. Best is trial 0 with value: 2.628667415046948e-05.
[I 2026-04-12 10:13:06,592] Trial 2 finished with value: 2.7207118255319073e-05 and parameters: {'lr': 0.000314733226669256, 'base_ch': 32, 'batch_size': 2}. Best is trial 0 with value: 2.628667415046948e-05.
[I 2026-04-12 10:16:54,546] Trial 3 finished with value: 4.510476836730959e-05 and parameters: {'lr': 0.0003198336815120354, 'base_ch': 24, 'batch_size': 2}. Best is trial 0 with value: 2.628667415046948e-05.
[I 2026-04-12 10:20:44,217] Trial 4 finished wit

Best trial:
Value: 2.628667415046948e-05
Params: {'lr': 0.00015917718036895313, 'base_ch': 32, 'batch_size': 2}


### Step 14 — Save Optuna results

In [31]:
# Step 14: Save Optuna results

optuna_rows = []

for t in study.trials:
    optuna_rows.append({
        "trial_number": t.number,
        "value": t.value,
        "state": str(t.state),
        **t.params
    })

optuna_df = pd.DataFrame(optuna_rows)
optuna_csv_path = OUT_DIR / "optuna_trials.csv"
optuna_df.to_csv(optuna_csv_path, index=False)

print("Saved:", optuna_csv_path)
optuna_df.sort_values("value", ascending=True).head()

Saved: /content/drive/MyDrive/rectified_flow_ct2dose/analysis/phase3_wandb_optuna/optuna_trials.csv


,trial_number,value,state,lr,base_ch,batch_size
0,0,0.000026,TrialState.COMPLETE,0.000159,32,2
2,2,0.000027,TrialState.COMPLETE,0.000315,32,2
1,1,0.000037,TrialState.COMPLETE,0.000288,32,2
4,4,0.000038,TrialState.COMPLETE,0.000477,32,2
3,3,0.000045,TrialState.COMPLETE,0.000320,24,2


### Step 15 — Build final config from Optuna best trial

In [32]:
# Step 15: Build final config from Optuna best trial

best_params = study.best_trial.params

final_config = {
    "run_name": f"optuna_best_lr{best_params['lr']:.1e}_base{best_params['base_ch']}_bs{best_params['batch_size']}_ep50",
    "lr": best_params["lr"],
    "base_ch": best_params["base_ch"],
    "batch_size": best_params["batch_size"],
    "epochs": 50,
}

final_config

{'run_name': 'optuna_best_lr1.6e-04_base32_bs2_ep50',
 'lr': 0.00015917718036895313,
 'base_ch': 32,
 'batch_size': 2,
 'epochs': 50}

### Step 16 — Train final selected config with wandb

In [33]:
# Step 16: Train final selected config with wandb

final_summary = run_one_wandb_experiment(
    final_config,
    project_name="ct2dose-phase3-optuna-selected"
)

final_summary

best_epoch,▁
best_val_loss,▁
elapsed_min,▁
epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
final_val_mae,▁
final_val_mse,▁
train_loss,█▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▄▃▂▂▂▁▁▄▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▂▁▁▁▁▁▁▁▁▁▂▁▂
best_ckpt_path,/content/drive/MyDri...
best_epoch,44
best_val_loss,1e-05


{'best_epoch': 44,
 'best_val_loss': 1.097921534346824e-05,
 'final_val_mse': 1.3778796175756724e-05,
 'final_val_mae': 0.002420939379371703,
 'elapsed_min': 10.127114061514536,
 'best_ckpt_path': '/content/drive/MyDrive/rectified_flow_ct2dose/outputs/checkpoints/optuna_best_lr1.6e-04_base32_bs2_ep50_best.pt',
 'latest_ckpt_path': '/content/drive/MyDrive/rectified_flow_ct2dose/outputs/checkpoints/optuna_best_lr1.6e-04_base32_bs2_ep50_latest.pt'}

### Step 17 — Save a markdown summary

In [34]:
# Step 17: Save markdown summary

summary_lines = []
summary_lines.append("# Wandb / Optuna Tuning Summary\n")
summary_lines.append("## Wandb")
summary_lines.append("- A first wandb-based tuning workflow was set up and tested.")
summary_lines.append("- Small controlled runs were logged successfully.\n")

summary_lines.append("## Optuna")
summary_lines.append(f"- Number of trials: {len(study.trials)}")
summary_lines.append(f"- Best value: {study.best_trial.value:.6e}")
summary_lines.append(f"- Best params: {study.best_trial.params}\n")

summary_lines.append("## Final selected run")
summary_lines.append(f"- run_name: `{final_config['run_name']}`")
summary_lines.append(f"- lr: `{final_config['lr']}`")
summary_lines.append(f"- base_ch: `{final_config['base_ch']}`")
summary_lines.append(f"- batch_size: `{final_config['batch_size']}`")
summary_lines.append(f"- epochs: `{final_config['epochs']}`")
summary_lines.append(f"- final summary: {final_summary}")

summary_md_path = OUT_DIR / "wandb_optuna_tuning_summary.md"
with open(summary_md_path, "w", encoding="utf-8") as f:
    f.write("\n".join(summary_lines))

print("Saved:", summary_md_path)
print("\n".join(summary_lines))

Saved: /content/drive/MyDrive/rectified_flow_ct2dose/analysis/phase3_wandb_optuna/wandb_optuna_tuning_summary.md
# Wandb / Optuna Tuning Summary

## Wandb
- A first wandb-based tuning workflow was set up and tested.
- Small controlled runs were logged successfully.

## Optuna
- Number of trials: 8
- Best value: 2.628667e-05
- Best params: {'lr': 0.00015917718036895313, 'base_ch': 32, 'batch_size': 2}

## Final selected run
- run_name: `optuna_best_lr1.6e-04_base32_bs2_ep50`
- lr: `0.00015917718036895313`
- base_ch: `32`
- batch_size: `2`
- epochs: `50`
- final summary: {'best_epoch': 44, 'best_val_loss': 1.097921534346824e-05, 'final_val_mse': 1.3778796175756724e-05, 'final_val_mae': 0.002420939379371703, 'elapsed_min': 10.127114061514536, 'best_ckpt_path': '/content/drive/MyDrive/rectified_flow_ct2dose/outputs/checkpoints/optuna_best_lr1.6e-04_base32_bs2_ep50_best.pt', 'latest_ckpt_path': '/content/drive/MyDrive/rectified_flow_ct2dose/outputs/checkpoints/optuna_best_lr1.6e-04_base32_b

In [35]:
# Final selected Optuna configuration (full 50-epoch run)

final_optuna_config = {
    "run_name": "optuna_final_lr4p2e4_base24_bs2_ep50",
    "lr": 4.199453944272278e-4,
    "base_ch": 24,
    "batch_size": 2,
    "epochs": 50,
}

final_optuna_summary = run_one_wandb_experiment(
    final_optuna_config,
    project_name="ct2dose-phase3-optuna-selected"
)

final_optuna_summary

best_epoch,▁
best_val_loss,▁
elapsed_min,▁
epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇███
final_val_mae,▁
final_val_mse,▁
train_loss,█▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▅▄▃▂▂▆▃▂▂▁▂▂▁▁▂▁▁▂▃▃▃▁▂▁▂▁▁▁▁▁▁▂▁▁▂▁▁▁▂
best_ckpt_path,/content/drive/MyDri...
best_epoch,40
best_val_loss,2e-05


{'best_epoch': 40,
 'best_val_loss': 1.9576001868699677e-05,
 'final_val_mse': 2.4027183688303922e-05,
 'final_val_mae': 0.003285620872862637,
 'elapsed_min': 9.739909756183625,
 'best_ckpt_path': '/content/drive/MyDrive/rectified_flow_ct2dose/outputs/checkpoints/optuna_final_lr4p2e4_base24_bs2_ep50_best.pt',
 'latest_ckpt_path': '/content/drive/MyDrive/rectified_flow_ct2dose/outputs/checkpoints/optuna_final_lr4p2e4_base24_bs2_ep50_latest.pt'}